In [ ]:
!git clone https://github.com/Saahil-doryu/satellite-image-change-detection.git

Cloning into 'satellite-image-change-detection'...
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 16 (delta 4), reused 7 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (16/16), 880.98 KiB | 9.27 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [ ]:
!cd satellite-image-change-detection/

In [ ]:
import os
import torch
import pandas as pd

print("Python environment ready")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )


Python environment ready
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
!pip install -q -U "transformers>=4.50.0" accelerate datasets pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 38.6 MB/s eta 0:00:00


In [ ]:
import transformers
import accelerate
import datasets

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)

Transformers: 5.17.0
Accelerate: 1.15.0
Datasets: 5.0.1


In [ ]:
!pip install -q --force-reinstall "Pillow==11.3.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 124.2 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "google/RSRCC",
    split="test",
    streaming=True
)

print("RSRCC test dataset loaded")

README.md:   0%|          | 0.00/5.61k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/12835 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2521 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/3653 [00:00<?, ?it/s]

RSRCC test dataset loaded


In [ ]:
sample = next(iter(dataset))

print(sample.keys())
print(sample["text"])
print("Before image:", sample["before"].size)
print("After image:", sample["after"].size)

dict_keys(['before', 'after', 'text'])
**Question:** In the eastern portion of the image, along the residential street that was developed, what change occurred to the sidewalk at that location?\n\n**A)** The sidewalk was removed to make room for a larger lawn.\n**B)** The sidewalk was newly constructed on previously undeveloped land.\n**C)** The sidewalk was widened to accommodate more foot traffic.\n**D)** The sidewalk was replaced with a gravel path.\n\n**Answer:** B
Before image: (512, 512)
After image: (512, 512)


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

model_name = "HuggingFaceTB/SmolVLM-500M-Instruct"

processor = AutoProcessor.from_pretrained(
    model_name
)

model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("SmolVLM loaded successfully")
print("Model device:", next(model.parameters()).device)


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.


...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.02GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

SmolVLM loaded successfully
Model device: cuda:0


In [ ]:
sample = next(iter(dataset))

before = sample["before"]
after = sample["after"]

text = sample["text"]

parts = text.split("**Answer:**", 1)

question = parts[0].replace("**Question:**", "").strip()
ground_truth = parts[1].strip() if len(parts) > 1 else ""

print("Question:")
print(question)

print("\nGround truth:")
print(ground_truth)

print("\nBefore:", before.size)
print("After:", after.size)

Question:
In the eastern portion of the image, along the residential street that was developed, what change occurred to the sidewalk at that location?\n\n**A)** The sidewalk was removed to make room for a larger lawn.\n**B)** The sidewalk was newly constructed on previously undeveloped land.\n**C)** The sidewalk was widened to accommodate more foot traffic.\n**D)** The sidewalk was replaced with a gravel path.\n\n

Ground truth:
B

Before: (512, 512)
After: (512, 512)


In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": before
            },
            {
                "type": "image",
                "image": after
            },
            {
                "type": "text",
                "text": f"""
Compare the BEFORE and AFTER satellite images.

Question:
{question}

Answer based on the changes between the two images.
Give a concise answer.
"""
            }
        ]
    }
]

prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(prompt[:1000])

<|im_start|>User:<image><image>
Compare the BEFORE and AFTER satellite images.

Question:
In the eastern portion of the image, along the residential street that was developed, what change occurred to the sidewalk at that location?\n\n**A)** The sidewalk was removed to make room for a larger lawn.\n**B)** The sidewalk was newly constructed on previously undeveloped land.\n**C)** The sidewalk was widened to accommodate more foot traffic.\n**D)** The sidewalk was replaced with a gravel path.\n\n

Answer based on the changes between the two images.
Give a concise answer.
<end_of_utterance>
Assistant:


In [ ]:
inputs = processor(
    text=prompt,
    images=[before, after],
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device) if hasattr(v, "to") else v
    for k, v in inputs.items()
}

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

generated_ids_trimmed = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(
        inputs["input_ids"],
        generated_ids
    )
]

prediction = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True
)[0].strip()

print("QUESTION:")
print(question)

print("\nGROUND TRUTH:")
print(ground_truth)

print("\nSMOLVLM PREDICTION:")
print(prediction)

QUESTION:
In the eastern portion of the image, along the residential street that was developed, what change occurred to the sidewalk at that location?\n\n**A)** The sidewalk was removed to make room for a larger lawn.\n**B)** The sidewalk was newly constructed on previously undeveloped land.\n**C)** The sidewalk was widened to accommodate more foot traffic.\n**D)** The sidewalk was replaced with a gravel path.\n\n

GROUND TRUTH:
B

SMOLVLM PREDICTION:
A.


In [ ]:
import re

def parse_rsrcc_text(text):
    parts = text.split("**Answer:**", 1)

    question = parts[0].replace(
        "**Question:**",
        ""
    ).strip()

    ground_truth = (
        parts[1].strip().upper()
        if len(parts) > 1
        else ""
    )

    return question, ground_truth


def get_question_type(question):

    if re.search(
        r"\*\*[A-D]\)\*\*|\b[A-D]\)",
        question,
        re.IGNORECASE
    ):
        return "multiple_choice"

    return "yes_no"


def extract_answer(prediction, question):

    if prediction is None:
        return None

    prediction = str(prediction).strip().upper()

    question_type = get_question_type(question)

    if question_type == "multiple_choice":

        match = re.search(
            r"\b([ABCD])\b",
            prediction
        )

        if match:
            return match.group(1)

    else:

        match = re.search(
            r"\b(YES|NO)\b",
            prediction
        )

        if match:
            return match.group(1)

    return None


# Test current prediction

question, ground_truth = parse_rsrcc_text(
    sample["text"]
)

question_type = get_question_type(
    question
)

prediction_answer = extract_answer(
    prediction,
    question
)

print("Question type:", question_type)
print("Ground truth:", ground_truth)
print("Raw prediction:", prediction)
print("Extracted answer:", prediction_answer)
print(
    "Correct:",
    prediction_answer == ground_truth
)

Question type: yes_no
Ground truth: YES
Raw prediction: No.
Extracted answer: NO
Correct: False


In [ ]:
results = []

for i, sample in enumerate(dataset):

    if i >= 20:
        break

    question, ground_truth = parse_rsrcc_text(
        sample["text"]
    )

    question_type = get_question_type(question)

    try:

        raw_prediction = run_smolvlm(
            sample["before"],
            sample["after"],
            question
        )

        prediction = extract_answer(
            raw_prediction,
            question
        )

        correct = (
            prediction == ground_truth
        )

        results.append({
            "sample_id": i,
            "question_type": question_type,
            "question": question,
            "ground_truth": ground_truth,
            "raw_prediction": raw_prediction,
            "prediction": prediction,
            "correct": correct
        })

        print(
            f"{i+1}/20 | "
            f"{question_type} | "
            f"GT={ground_truth} | "
            f"Pred={prediction} | "
            f"Correct={correct}"
        )

    except Exception as e:

        results.append({
            "sample_id": i,
            "question_type": question_type,
            "question": question,
            "ground_truth": ground_truth,
            "raw_prediction": "",
            "prediction": None,
            "correct": False,
            "error": str(e)
        })

        print(
            f"{i+1}/20 | ERROR | {e}"
        )

1/20 | multiple_choice | GT=B | Pred=A | Correct=False
2/20 | yes_no | GT=YES | Pred=NO | Correct=False
3/20 | multiple_choice | GT=B | Pred=A | Correct=False
4/20 | yes_no | GT=NO | Pred=NO | Correct=True
5/20 | multiple_choice | GT=B | Pred=A | Correct=False
6/20 | yes_no | GT=NO | Pred=YES | Correct=False
7/20 | yes_no | GT=YES | Pred=NO | Correct=False
8/20 | yes_no | GT=YES | Pred=NO | Correct=False
9/20 | multiple_choice | GT=A | Pred=A | Correct=True
10/20 | yes_no | GT=YES | Pred=NO | Correct=False
11/20 | yes_no | GT=YES | Pred=NO | Correct=False
12/20 | multiple_choice | GT=A | Pred=A | Correct=True
13/20 | yes_no | GT=YES | Pred=YES | Correct=True
14/20 | multiple_choice | GT=A | Pred=A | Correct=True
15/20 | multiple_choice | GT=B | Pred=A | Correct=False
16/20 | multiple_choice | GT=B | Pred=A | Correct=False
17/20 | yes_no | GT=YES | Pred=NO | Correct=False
18/20 | yes_no | GT=YES | Pred=NO | Correct=False
19/20 | multiple_choice | GT=B | Pred=A | Correct=False
20/20 | ye

In [ ]:
import os
import pandas as pd

results_dir = "/content/satellite-image-change-detection/results"

os.makedirs(results_dir, exist_ok=True)

diagnostic_file = os.path.join(
    results_dir,
    "task2_smolvlm_20_diagnostic.csv"
)

smol_test_df = pd.DataFrame(results)
smol_test_df.to_csv(
    diagnostic_file,
    index=False
)

print("Saved in notebook project:")
print(diagnostic_file)

Saved in notebook project:
/content/satellite-image-change-detection/results/task2_smolvlm_20_diagnostic.csv


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

model_name = "HuggingFaceTB/SmolVLM-500M-Instruct"

processor = AutoProcessor.from_pretrained(
    model_name
)

model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("SmolVLM loaded successfully")
print("Model device:", next(model.parameters()).device)


# Model 3: InternVL2.5-4B

In [ ]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

PyTorch: 2.11.0+cu128
Transformers: 5.17.0
CUDA: True
GPU: Tesla T4
GPU memory: 14.56 GB


# Qwen2.5-VL-7B evaluation

In [ ]:
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.2 MB/s eta 0:00:00


In [ ]:
import bitsandbytes as bnb

print("BitsAndBytes:", bnb.__version__)
print("BitsAndBytes is available.")

BitsAndBytes: 0.50.2
BitsAndBytes is available.


In [ ]:
import torch
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig
)

model_name = "Qwen/Qwen2.5-VL-7B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

processor_7b = AutoProcessor.from_pretrained(
    model_name
)

model_7b = AutoModelForImageTextToText.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16
)

model_7b.eval()

print("Qwen2.5-VL-7B loaded successfully")
print("Device:", next(model_7b.parameters()).device)

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Qwen2.5-VL-7B loaded successfully
Device: cuda:0


In [ ]:
import re

def parse_rsrcc_text(text):
    parts = text.split("**Answer:**", 1)

    question = parts[0].replace(
        "**Question:**",
        ""
    ).strip()

    ground_truth = (
        parts[1].strip().upper()
        if len(parts) > 1
        else ""
    )

    return question, ground_truth


def get_question_type(question):
    if re.search(
        r"\*\*[A-D]\)\*\*|\b[A-D]\)",
        question,
        re.IGNORECASE
    ):
        return "multiple_choice"

    return "yes_no"


def extract_answer(prediction, question):
    if prediction is None:
        return None

    prediction = str(prediction).strip().upper()

    if get_question_type(question) == "multiple_choice":
        match = re.search(r"\b([ABCD])\b", prediction)
    else:
        match = re.search(r"\b(YES|NO)\b", prediction)

    if match:
        return match.group(1)

    return None

print("Helper functions loaded.")

Helper functions loaded.


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "google/RSRCC",
    split="test",
    streaming=True
)

sample = next(iter(dataset))

before = sample["before"]
after = sample["after"]

question, ground_truth = parse_rsrcc_text(
    sample["text"]
)

print("QUESTION:")
print(question)

print("\nGROUND TRUTH:")
print(ground_truth)

print("\nBEFORE:", before.size)
print("AFTER:", after.size)

Resolving data files:   0%|          | 0/12835 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2521 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/3653 [00:00<?, ?it/s]

QUESTION:
In the eastern portion of the image, along the residential street that was developed, what change occurred to the sidewalk at that location?\n\n**A)** The sidewalk was removed to make room for a larger lawn.\n**B)** The sidewalk was newly constructed on previously undeveloped land.\n**C)** The sidewalk was widened to accommodate more foot traffic.\n**D)** The sidewalk was replaced with a gravel path.\n\n

GROUND TRUTH:
B

BEFORE: (512, 512)
AFTER: (512, 512)


In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": before
            },
            {
                "type": "image",
                "image": after
            },
            {
                "type": "text",
                "text": f"""
You are analyzing two satellite images of the same location.

The first image is BEFORE.
The second image is AFTER.

Compare the two images and answer the question based ONLY
on the changes between the images.

Question:
{question}

For a multiple-choice question, answer with only A, B, C, or D.

For a yes/no question, answer with only YES or NO.
"""
            }
        ]
    }
]

prompt = processor_7b.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = processor_7b(
    text=[prompt],
    images=[before, after],
    padding=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model_7b.device) if hasattr(v, "to") else v
    for k, v in inputs.items()
}

with torch.inference_mode():
    generated_ids = model_7b.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False
    )

generated_ids_trimmed = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(
        inputs["input_ids"],
        generated_ids
    )
]

prediction = processor_7b.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True
)[0].strip()

print("=" * 70)
print("QUESTION:")
print(question)

print("\nGROUND TRUTH:")
print(ground_truth)

print("\nQWEN2.5-VL-7B PREDICTION:")
print(prediction)

print("=" * 70)

QUESTION:
In the eastern portion of the image, along the residential street that was developed, what change occurred to the sidewalk at that location?\n\n**A)** The sidewalk was removed to make room for a larger lawn.\n**B)** The sidewalk was newly constructed on previously undeveloped land.\n**C)** The sidewalk was widened to accommodate more foot traffic.\n**D)** The sidewalk was replaced with a gravel path.\n\n

GROUND TRUTH:
B

QWEN2.5-VL-7B PREDICTION:
B


In [ ]:
prediction_answer = extract_answer(
    prediction,
    question
)

print("Question type:", get_question_type(question))
print("Ground truth:", ground_truth)
print("Raw prediction:", prediction)
print("Extracted answer:", prediction_answer)
print("Correct:", prediction_answer == ground_truth)

Question type: multiple_choice
Ground truth: B
Raw prediction: B
Extracted answer: B
Correct: True


In [ ]:
results_7b = []

dataset_7b = load_dataset(
    "google/RSRCC",
    split="test",
    streaming=True
)

for i, sample in enumerate(dataset_7b):

    if i >= 20:
        break

    before = sample["before"]
    after = sample["after"]

    question, ground_truth = parse_rsrcc_text(
        sample["text"]
    )

    question_type = get_question_type(question)

    try:

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": before
                    },
                    {
                        "type": "image",
                        "image": after
                    },
                    {
                        "type": "text",
                        "text": f"""
You are analyzing two satellite images of the same location.

The first image is BEFORE.
The second image is AFTER.

Compare the two images and answer the question based ONLY
on the changes between the images.

Question:
{question}

For a multiple-choice question, answer with only A, B, C, or D.

For a yes/no question, answer with only YES or NO.
"""
                    }
                ]
            }
        ]

        prompt = processor_7b.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = processor_7b(
            text=[prompt],
            images=[before, after],
            padding=True,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(model_7b.device)
            if hasattr(v, "to") else v
            for k, v in inputs.items()
        }

        with torch.inference_mode():

            generated_ids = model_7b.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False
            )

        generated_ids_trimmed = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(
                inputs["input_ids"],
                generated_ids
            )
        ]

        raw_prediction = processor_7b.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True
        )[0].strip()

        prediction = extract_answer(
            raw_prediction,
            question
        )

        correct = prediction == ground_truth

        results_7b.append({
            "sample_id": i,
            "question_type": question_type,
            "question": question,
            "ground_truth": ground_truth,
            "raw_prediction": raw_prediction,
            "prediction": prediction,
            "correct": correct
        })

        print(
            f"{i+1}/20 | "
            f"{question_type} | "
            f"GT={ground_truth} | "
            f"Pred={prediction} | "
            f"Correct={correct}"
        )

    except Exception as e:

        results_7b.append({
            "sample_id": i,
            "question_type": question_type,
            "question": question,
            "ground_truth": ground_truth,
            "raw_prediction": "",
            "prediction": None,
            "correct": False,
            "error": str(e)
        })

        print(
            f"{i+1}/20 | ERROR | {e}"
        )

print("\n20-sample evaluation complete.")

Resolving data files:   0%|          | 0/12835 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2521 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/3653 [00:00<?, ?it/s]

1/20 | multiple_choice | GT=B | Pred=B | Correct=True
2/20 | yes_no | GT=YES | Pred=YES | Correct=True
3/20 | multiple_choice | GT=B | Pred=B | Correct=True
4/20 | yes_no | GT=NO | Pred=NO | Correct=True
5/20 | multiple_choice | GT=B | Pred=B | Correct=True
6/20 | yes_no | GT=NO | Pred=NO | Correct=True
7/20 | yes_no | GT=YES | Pred=NO | Correct=False
8/20 | yes_no | GT=YES | Pred=NO | Correct=False
9/20 | multiple_choice | GT=A | Pred=B | Correct=False
10/20 | yes_no | GT=YES | Pred=NO | Correct=False
11/20 | yes_no | GT=YES | Pred=NO | Correct=False
12/20 | multiple_choice | GT=A | Pred=B | Correct=False
13/20 | yes_no | GT=YES | Pred=NO | Correct=False
14/20 | multiple_choice | GT=A | Pred=B | Correct=False
15/20 | multiple_choice | GT=B | Pred=B | Correct=True
16/20 | multiple_choice | GT=B | Pred=B | Correct=True
17/20 | yes_no | GT=YES | Pred=NO | Correct=False
18/20 | yes_no | GT=YES | Pred=NO | Correct=False
19/20 | multiple_choice | GT=B | Pred=B | Correct=True
20/20 | yes_no 

In [ ]:
import pandas as pd

results_7b_df = pd.DataFrame(results_7b)

total = len(results_7b_df)
correct = int(results_7b_df["correct"].sum())

print("=" * 60)
print("QWEN2.5-VL-7B - 20 SAMPLE RESULTS")
print("=" * 60)

print("Total:", total)
print("Correct:", correct)
print("Incorrect:", total - correct)

print(
    "Overall Accuracy:",
    round(correct / total * 100, 2),
    "%"
)

QWEN2.5-VL-7B - 20 SAMPLE RESULTS
Total: 20
Correct: 10
Incorrect: 10
Overall Accuracy: 50.0 %


In [ ]:
yes_no_7b = results_7b_df[
    results_7b_df["question_type"] == "yes_no"
]

mc_7b = results_7b_df[
    results_7b_df["question_type"] == "multiple_choice"
]

print("\nYES/NO")
print("Total:", len(yes_no_7b))

if len(yes_no_7b) > 0:
    print(
        "Correct:",
        int(yes_no_7b["correct"].sum())
    )
    print(
        "Accuracy:",
        round(
            yes_no_7b["correct"].mean() * 100,
            2
        ),
        "%"
    )

print("\nMULTIPLE CHOICE")
print("Total:", len(mc_7b))

if len(mc_7b) > 0:
    print(
        "Correct:",
        int(mc_7b["correct"].sum())
    )
    print(
        "Accuracy:",
        round(
            mc_7b["correct"].mean() * 100,
            2
        ),
        "%"
    )


YES/NO
Total: 11
Correct: 4
Accuracy: 36.36 %

MULTIPLE CHOICE
Total: 9
Correct: 6
Accuracy: 66.67 %


In [ ]:
import os

results_dir = "/content/satellite-image-change-detection/results"

os.makedirs(results_dir, exist_ok=True)

file_7b = os.path.join(
    results_dir,
    "task2_qwen25_vl_7b_20_results.csv"
)

results_7b_df.to_csv(
    file_7b,
    index=False
)

print("Saved:")
print(file_7b)

Saved:
/content/satellite-image-change-detection/results/task2_qwen25_vl_7b_20_results.csv


In [ ]:
# Display all 20 predictions

display(
    results_7b_df[
        [
            "sample_id",
            "question_type",
            "ground_truth",
            "raw_prediction",
            "prediction",
            "correct"
        ]
    ]
)

,sample_id,question_type,ground_truth,raw_prediction,prediction,correct
0,0,multiple_choice,B,B,B,True
1,1,yes_no,YES,YES,YES,True
2,2,multiple_choice,B,B,B,True
3,3,yes_no,NO,NO,NO,True
4,4,multiple_choice,B,B,B,True
5,5,yes_no,NO,NO,NO,True
6,6,yes_no,YES,NO,NO,False
7,7,yes_no,YES,NO,NO,False
8,8,multiple_choice,A,B,B,False
9,9,yes_no,YES,NO,NO,False


In [ ]:
print("=" * 60)
print("TASK 2 MODEL COMPARISON")
print("=" * 60)

print(f"Qwen2.5-VL-3B:     81.8%  (1000 samples)")
print(f"SmolVLM-500M:      25.0%  (20 samples)")

if len(results_7b_df) > 0:
    accuracy_7b = results_7b_df["correct"].mean() * 100
    print(
        f"Qwen2.5-VL-7B:     {accuracy_7b:.2f}%  (20 samples)"
    )

TASK 2 MODEL COMPARISON
Qwen2.5-VL-3B:     81.8%  (1000 samples)
SmolVLM-500M:      25.0%  (20 samples)
Qwen2.5-VL-7B:     50.00%  (20 samples)


In [ ]:
wrong_7b = results_7b_df[
    results_7b_df["correct"] == False
]

print("Incorrect predictions:", len(wrong_7b))

display(
    wrong_7b[
        [
            "sample_id",
            "question_type",
            "ground_truth",
            "raw_prediction",
            "prediction"
        ]
    ]
)

Incorrect predictions: 10


,sample_id,question_type,ground_truth,raw_prediction,prediction
6,6,yes_no,YES,NO,NO
7,7,yes_no,YES,NO,NO
8,8,multiple_choice,A,B,B
9,9,yes_no,YES,NO,NO
10,10,yes_no,YES,NO,NO
11,11,multiple_choice,A,B,B
12,12,yes_no,YES,NO,NO
13,13,multiple_choice,A,B,B
16,16,yes_no,YES,NO,NO
17,17,yes_no,YES,NO,NO


In [ ]:
qwen100_results = []

dataset_7b_100 = load_dataset(
    "google/RSRCC",
    split="test",
    streaming=True
)

for i, sample in enumerate(dataset_7b_100):

    if i >= 100:
        break

    before = sample["before"]
    after = sample["after"]

    question, ground_truth = parse_rsrcc_text(
        sample["text"]
    )

    question_type = get_question_type(question)

    try:

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": before
                    },
                    {
                        "type": "image",
                        "image": after
                    },
                    {
                        "type": "text",
                        "text": f"""
You are analyzing two satellite images of the same location.

The first image is BEFORE.
The second image is AFTER.

Compare the two images and answer the question based ONLY
on the changes between the images.

Question:
{question}

For a multiple-choice question, answer with only A, B, C, or D.

For a yes/no question, answer with only YES or NO.
"""
                    }
                ]
            }
        ]

        prompt = processor_7b.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = processor_7b(
            text=[prompt],
            images=[before, after],
            padding=True,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(model_7b.device)
            if hasattr(v, "to") else v
            for k, v in inputs.items()
        }

        with torch.inference_mode():

            generated_ids = model_7b.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False
            )

        generated_ids_trimmed = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(
                inputs["input_ids"],
                generated_ids
            )
        ]

        raw_prediction = processor_7b.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True
        )[0].strip()

        prediction = extract_answer(
            raw_prediction,
            question
        )

        correct = prediction == ground_truth

        qwen100_results.append({
            "sample_id": i,
            "question_type": question_type,
            "question": question,
            "ground_truth": ground_truth,
            "raw_prediction": raw_prediction,
            "prediction": prediction,
            "correct": correct
        })

        print(
            f"{i+1}/100 | "
            f"{question_type} | "
            f"GT={ground_truth} | "
            f"Pred={prediction} | "
            f"Correct={correct}"
        )

    except Exception as e:

        qwen100_results.append({
            "sample_id": i,
            "question_type": question_type,
            "question": question,
            "ground_truth": ground_truth,
            "raw_prediction": "",
            "prediction": None,
            "correct": False,
            "error": str(e)
        })

        print(f"{i+1}/100 | ERROR | {e}")

Resolving data files:   0%|          | 0/12835 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2521 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/3653 [00:00<?, ?it/s]

1/100 | multiple_choice | GT=B | Pred=B | Correct=True
2/100 | yes_no | GT=YES | Pred=YES | Correct=True
3/100 | multiple_choice | GT=B | Pred=B | Correct=True
4/100 | yes_no | GT=NO | Pred=NO | Correct=True
5/100 | multiple_choice | GT=B | Pred=B | Correct=True
6/100 | yes_no | GT=NO | Pred=NO | Correct=True
7/100 | yes_no | GT=YES | Pred=NO | Correct=False
8/100 | yes_no | GT=YES | Pred=NO | Correct=False
9/100 | multiple_choice | GT=A | Pred=B | Correct=False
10/100 | yes_no | GT=YES | Pred=NO | Correct=False
11/100 | yes_no | GT=YES | Pred=NO | Correct=False
12/100 | multiple_choice | GT=A | Pred=B | Correct=False
13/100 | yes_no | GT=YES | Pred=NO | Correct=False
14/100 | multiple_choice | GT=A | Pred=B | Correct=False
15/100 | multiple_choice | GT=B | Pred=B | Correct=True
16/100 | multiple_choice | GT=B | Pred=B | Correct=True
17/100 | yes_no | GT=YES | Pred=NO | Correct=False
18/100 | yes_no | GT=YES | Pred=NO | Correct=False
19/100 | multiple_choice | GT=B | Pred=B | Correct=T

In [ ]:
total = len(final_results_100)

correct = int(
    final_results_100["correct"].sum()
)

incorrect = total - correct

accuracy = (
    correct / total * 100
)

print("=" * 60)
print("QWEN2.5-VL-7B — 100 SAMPLES")
print("=" * 60)
print("Total:", total)
print("Correct:", correct)
print("Incorrect:", incorrect)
print("Accuracy:", round(accuracy, 2), "%")

QWEN2.5-VL-7B — 100 SAMPLES
Total: 100
Correct: 57
Incorrect: 43
Accuracy: 57.0 %


In [ ]:
print("\nQUESTION TYPE RESULTS")

for qtype in ["yes_no", "multiple_choice"]:

    df = final_results_100[
        final_results_100["question_type"] == qtype
    ]

    print("\n", qtype)

    print("Total:", len(df))
    print("Correct:", int(df["correct"].sum()))
    print(
        "Accuracy:",
        round(df["correct"].mean() * 100, 2),
        "%"
    )


QUESTION TYPE RESULTS

 yes_no
Total: 47
Correct: 17
Accuracy: 36.17 %

 multiple_choice
Total: 53
Correct: 40
Accuracy: 75.47 %


In [ ]:
path = "/content/satellite-image-change-detection/results/qwen2.5_vl_7b_100_results.csv"

final_results_100.to_csv(
    path,
    index=False
)

print("Saved:", path)

Saved: /content/satellite-image-change-detection/results/qwen2.5_vl_7b_100_results.csv


In [ ]:
import pandas as pd

qwen100_df = pd.DataFrame(qwen100_results)

total = len(qwen100_df)
correct = int(qwen100_df["correct"].sum())
incorrect = total - correct
accuracy = correct / total * 100

yes_no_df = qwen100_df[
    qwen100_df["question_type"] == "yes_no"
]

mc_df = qwen100_df[
    qwen100_df["question_type"] == "multiple_choice"
]

summary_100 = pd.DataFrame({
    "Model": ["Qwen2.5-VL-7B"],
    "Samples": [total],
    "Correct": [correct],
    "Incorrect": [incorrect],
    "Overall_Accuracy": [round(accuracy, 2)],
    "Yes_No_Samples": [len(yes_no_df)],
    "Yes_No_Accuracy": [
        round(yes_no_df["correct"].mean() * 100, 2)
    ],
    "Multiple_Choice_Samples": [len(mc_df)],
    "Multiple_Choice_Accuracy": [
        round(mc_df["correct"].mean() * 100, 2)
    ]
})

print(summary_100)

NameError: name 'qwen100_results' is not defined

In [ ]:
qwen1000_results = []

dataset_7b_1000 = load_dataset(
    "google/RSRCC",
    split="test",
    streaming=True
)

for i, sample in enumerate(dataset_7b_1000):

    if i >= 1000:
        break

    before = sample["before"]
    after = sample["after"]

    question, ground_truth = parse_rsrcc_text(
        sample["text"]
    )

    question_type = get_question_type(question)

    try:

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": before
                    },
                    {
                        "type": "image",
                        "image": after
                    },
                    {
                        "type": "text",
                        "text": f"""
You are analyzing two satellite images of the same location.

The first image is BEFORE.
The second image is AFTER.

Compare the two images and answer the question based ONLY
on the changes between the images.

Question:
{question}

For a multiple-choice question, answer with only A, B, C, or D.

For a yes/no question, answer with only YES or NO.
"""
                    }
                ]
            }
        ]

        prompt = processor_7b.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = processor_7b(
            text=[prompt],
            images=[before, after],
            padding=True,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(model_7b.device)
            if hasattr(v, "to") else v
            for k, v in inputs.items()
        }

        with torch.inference_mode():

            generated_ids = model_7b.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False
            )

        generated_ids_trimmed = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(
                inputs["input_ids"],
                generated_ids
            )
        ]

        raw_prediction = processor_7b.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True
        )[0].strip()

        prediction = extract_answer(
            raw_prediction,
            question
        )

        correct = prediction == ground_truth

        qwen1000_results.append({
            "sample_id": i,
            "question_type": question_type,
            "question": question,
            "ground_truth": ground_truth,
            "raw_prediction": raw_prediction,
            "prediction": prediction,
            "correct": correct
        })

        print(
            f"{i+1}/1000 | "
            f"{question_type} | "
            f"GT={ground_truth} | "
            f"Pred={prediction} | "
            f"Correct={correct}"
        )

    except Exception as e:

        qwen1000_results.append({
            "sample_id": i,
            "question_type": question_type,
            "question": question,
            "ground_truth": ground_truth,
            "raw_prediction": "",
            "prediction": None,
            "correct": False,
            "error": str(e)
        })

        print(f"{i+1}/1000 | ERROR | {e}")

Resolving data files:   0%|          | 0/12835 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2521 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/3653 [00:00<?, ?it/s]

1/1000 | multiple_choice | GT=B | Pred=B | Correct=True
2/1000 | yes_no | GT=YES | Pred=YES | Correct=True
3/1000 | multiple_choice | GT=B | Pred=B | Correct=True
4/1000 | yes_no | GT=NO | Pred=NO | Correct=True
5/1000 | multiple_choice | GT=B | Pred=B | Correct=True
6/1000 | yes_no | GT=NO | Pred=NO | Correct=True
7/1000 | yes_no | GT=YES | Pred=NO | Correct=False
8/1000 | yes_no | GT=YES | Pred=NO | Correct=False
9/1000 | multiple_choice | GT=A | Pred=B | Correct=False
10/1000 | yes_no | GT=YES | Pred=NO | Correct=False
11/1000 | yes_no | GT=YES | Pred=NO | Correct=False
12/1000 | multiple_choice | GT=A | Pred=B | Correct=False
13/1000 | yes_no | GT=YES | Pred=NO | Correct=False
14/1000 | multiple_choice | GT=A | Pred=B | Correct=False
15/1000 | multiple_choice | GT=B | Pred=B | Correct=True
16/1000 | multiple_choice | GT=B | Pred=B | Correct=True
17/1000 | yes_no | GT=YES | Pred=NO | Correct=False
18/1000 | yes_no | GT=YES | Pred=NO | Correct=False
19/1000 | multiple_choice | GT=B |

In [ ]:
import pandas as pd

qwen1000_df = pd.DataFrame(qwen1000_results)

total = len(qwen1000_df)
correct = int(qwen1000_df["correct"].sum())
incorrect = total - correct
accuracy = correct / total * 100

print("=" * 60)
print("QWEN2.5-VL-7B — 1000 SAMPLE RESULTS")
print("=" * 60)
print("Total:", total)
print("Correct:", correct)
print("Incorrect:", incorrect)
print("Accuracy:", round(accuracy, 2), "%")

QWEN2.5-VL-7B — 1000 SAMPLE RESULTS
Total: 1000
Correct: 512
Incorrect: 488
Accuracy: 51.2 %


In [ ]:
yes_no_df = qwen1000_df[
    qwen1000_df["question_type"] == "yes_no"
]

mc_df = qwen1000_df[
    qwen1000_df["question_type"] == "multiple_choice"
]

print("\nYES/NO")
print("Total:", len(yes_no_df))
print("Correct:", int(yes_no_df["correct"].sum()))
print(
    "Accuracy:",
    round(yes_no_df["correct"].mean() * 100, 2),
    "%"
)

print("\nMULTIPLE CHOICE")
print("Total:", len(mc_df))
print("Correct:", int(mc_df["correct"].sum()))
print(
    "Accuracy:",
    round(mc_df["correct"].mean() * 100, 2),
    "%"
)


YES/NO
Total: 525
Correct: 188
Accuracy: 35.81 %

MULTIPLE CHOICE
Total: 475
Correct: 324
Accuracy: 68.21 %


In [ ]:
print("\nPREDICTION DISTRIBUTION")
print(qwen1000_df["prediction"].value_counts())

print("\nGROUND TRUTH DISTRIBUTION")
print(qwen1000_df["ground_truth"].value_counts())


PREDICTION DISTRIBUTION
prediction
NO     362
B      277
YES    163
A      118
C       45
D       35
Name: count, dtype: int64

GROUND TRUTH DISTRIBUTION
ground_truth
YES    500
B      249
A      195
NO      25
C       23
B)       4
A)       3
C)       1
Name: count, dtype: int64


In [ ]:
csv_path = "/content/qwen2.5_vl_7b_1000_results.csv"

qwen1000_df.to_csv(
    csv_path,
    index=False
)

print("CSV saved to:")
print(csv_path)

CSV saved to:
/content/qwen2.5_vl_7b_1000_results.csv


In [ ]:
summary = pd.DataFrame({
    "Model": ["Qwen2.5-VL-7B"],
    "Samples": [total],
    "Correct": [correct],
    "Incorrect": [incorrect],
    "Overall_Accuracy": [round(accuracy, 2)],
    "Yes_No_Samples": [len(yes_no_df)],
    "Yes_No_Accuracy": [round(yes_no_df["correct"].mean() * 100, 2)],
    "Multiple_Choice_Samples": [len(mc_df)],
    "Multiple_Choice_Accuracy": [round(mc_df["correct"].mean() * 100, 2)]
})

summary_path = "/content/qwen2.5_vl_7b_1000_summary.csv"

summary.to_csv(
    summary_path,
    index=False
)

print(summary)
print("\nSummary saved to:")
print(summary_path)

           Model  Samples  Correct  Incorrect  Overall_Accuracy  \
0  Qwen2.5-VL-7B     1000      512        488              51.2   

   Yes_No_Samples  Yes_No_Accuracy  Multiple_Choice_Samples  \
0             525            35.81                      475   

   Multiple_Choice_Accuracy  
0                     68.21  

Summary saved to:
/content/qwen2.5_vl_7b_1000_summary.csv


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

drive_project = "/content/drive/MyDrive/satellite_project"

print(os.listdir(drive_project))

['task1_1000_results.csv', 'task1_100_results.csv', 'task2_smolvlm_20_results.csv', 'qwen2.5_vl_7b_100_results.csv', 'qwen2.5_vl_7b_1000_results.csv', 'task2_qwen25_vl_7b_20_results.csv']


In [ ]:
import pandas as pd

task1_1000 = pd.read_csv(
    "/content/drive/MyDrive/satellite_project/task1_1000_results.csv"
)

qwen7b_1000 = pd.read_csv(
    "/content/drive/MyDrive/satellite_project/qwen2.5_vl_7b_1000_results.csv"
)

print("Task 1 rows:", len(task1_1000))
print("Qwen 7B rows:", len(qwen7b_1000))

display(task1_1000.head())
display(qwen7b_1000.head())

Task 1 rows: 1000
Qwen 7B rows: 1000


,sample,question,ground_truth,raw_prediction,prediction,correct,question_type
0,1,"In the western section of the neighborhood, ha...",Yes,Yes,yes,True,Yes/No
1,2,"In the northeast section of the property, what...",A,A,a,True,Multiple Choice
2,3,Regarding the front yard of the property locat...,B,B,b,True,Multiple Choice
3,4,What change is visible to the unpaved road loc...,A,A,a,True,Multiple Choice
4,5,Considering the overall configuration and layo...,B),B,b,True,Multiple Choice


,sample_id,question_type,question,ground_truth,raw_prediction,prediction,correct
0,0,multiple_choice,"In the eastern portion of the image, along the...",B,B,B,True
1,1,yes_no,Has the sidewalk at the northeast corner of th...,YES,YES,YES,True
2,2,multiple_choice,"Between the two images, what change occurred t...",B,B,B,True
3,3,yes_no,Is there any detectable anomaly or global alte...,NO,NO,NO,True
4,4,multiple_choice,What change can be observed regarding the vege...,B,B,B,True


In [ ]:
task1_accuracy = task1_1000["correct"].mean() * 100
qwen7b_accuracy = qwen7b_1000["correct"].mean() * 100

print("=" * 60)
print("TASK 2 MODEL COMPARISON")
print("=" * 60)

print(f"Qwen2.5-VL-3B: {task1_accuracy:.2f}%")
print(f"Qwen2.5-VL-7B: {qwen7b_accuracy:.2f}%")

print("\nDifference:")
print(f"{task1_accuracy - qwen7b_accuracy:.2f} percentage points")

TASK 2 MODEL COMPARISON
Qwen2.5-VL-3B: 81.80%
Qwen2.5-VL-7B: 51.20%

Difference:
30.60 percentage points


In [ ]:
for name, df in [
    ("Qwen2.5-VL-3B", task1_1000),
    ("Qwen2.5-VL-7B", qwen7b_1000)
]:
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    for qtype in df["question_type"].unique():
        subset = df[df["question_type"] == qtype]

        accuracy = subset["correct"].mean() * 100

        print(
            f"{qtype}: "
            f"{accuracy:.2f}% "
            f"({len(subset)} samples)"
        )


Qwen2.5-VL-3B
Yes/No: 91.73% (520 samples)
Multiple Choice: 71.04% (480 samples)

Qwen2.5-VL-7B
multiple_choice: 68.21% (475 samples)
yes_no: 35.81% (525 samples)


In [ ]:
print("=" * 60)
print("QWEN2.5-VL-3B PREDICTION DISTRIBUTION")
print("=" * 60)

print(task1_1000["prediction"].value_counts())

print("\nGround Truth:")
print(task1_1000["ground_truth"].value_counts())


print("\n" + "=" * 60)
print("QWEN2.5-VL-7B PREDICTION DISTRIBUTION")
print("=" * 60)

print(qwen7b_1000["prediction"].value_counts())

print("\nGround Truth:")
print(qwen7b_1000["ground_truth"].value_counts())

QWEN2.5-VL-3B PREDICTION DISTRIBUTION
prediction
yes    483
b      231
a      182
no      37
c       36
d       31
Name: count, dtype: int64

Ground Truth:
ground_truth
Yes    508
B      263
A      194
C       16
No      12
B)       4
A)       2
C)       1
Name: count, dtype: int64

QWEN2.5-VL-7B PREDICTION DISTRIBUTION
prediction
NO     362
B      277
YES    163
A      118
C       45
D       35
Name: count, dtype: int64

Ground Truth:
ground_truth
YES    500
B      249
A      195
NO      25
C       23
B)       4
A)       3
C)       1
Name: count, dtype: int64


In [ ]:
qwen3b_yesno = task1_1000[
    task1_1000["question_type"] == "Yes/No"
]

qwen7b_yesno = qwen7b_1000[
    qwen7b_1000["question_type"] == "yes_no"
]

print("=" * 60)
print("YES/NO RESPONSE BIAS")
print("=" * 60)

print("\nQwen2.5-VL-3B")
print(qwen3b_yesno["prediction"].value_counts(normalize=True).mul(100).round(2))

print("\nQwen2.5-VL-7B")
print(qwen7b_yesno["prediction"].value_counts(normalize=True).mul(100).round(2))

YES/NO RESPONSE BIAS

Qwen2.5-VL-3B
prediction
yes    92.88
no      7.12
Name: proportion, dtype: float64

Qwen2.5-VL-7B
prediction
NO     68.95
YES    31.05
Name: proportion, dtype: float64


In [ ]:
print("=" * 60)
print("ERROR ANALYSIS")
print("=" * 60)

for name, df in [
    ("Qwen2.5-VL-3B", task1_1000),
    ("Qwen2.5-VL-7B", qwen7b_1000)
]:

    print("\n" + name)

    for qtype in df["question_type"].unique():

        subset = df[df["question_type"] == qtype]

        total = len(subset)
        correct = subset["correct"].sum()
        incorrect = total - correct
        error_rate = incorrect / total * 100

        print(f"\n{qtype}")
        print("Total:", total)
        print("Correct:", correct)
        print("Incorrect:", incorrect)
        print("Error rate:", round(error_rate, 2), "%")

ERROR ANALYSIS

Qwen2.5-VL-3B

Yes/No
Total: 520
Correct: 477
Incorrect: 43
Error rate: 8.27 %

Multiple Choice
Total: 480
Correct: 341
Incorrect: 139
Error rate: 28.96 %

Qwen2.5-VL-7B

multiple_choice
Total: 475
Correct: 324
Incorrect: 151
Error rate: 31.79 %

yes_no
Total: 525
Correct: 188
Incorrect: 337
Error rate: 64.19 %


In [ ]:
print("=" * 70)
print("QWEN2.5-VL-3B ERROR PATTERNS")
print("=" * 70)

errors_3b = task1_1000[
    task1_1000["correct"] == False
]

print("\nGround truth vs prediction:")
print(
    pd.crosstab(
        errors_3b["ground_truth"],
        errors_3b["prediction"]
    )
)


print("\n" + "=" * 70)
print("QWEN2.5-VL-7B ERROR PATTERNS")
print("=" * 70)

errors_7b = qwen7b_1000[
    qwen7b_1000["correct"] == False
]

print("\nGround truth vs prediction:")
print(
    pd.crosstab(
        errors_7b["ground_truth"],
        errors_7b["prediction"]
    )
)

QWEN2.5-VL-3B ERROR PATTERNS

Ground truth vs prediction:
prediction     a   b   c   d  no  yes
ground_truth                         
A              0  35  12  14   0    0
A)             0   1   0   0   0    0
B             46   0  11  16   0    0
C              2   1   0   1   0    0
No             0   0   0   0   0    9
Yes            0   0   0   0  34    0

QWEN2.5-VL-7B ERROR PATTERNS

Ground truth vs prediction:
prediction     A   B   C   D   NO
ground_truth                     
A              0  63  16  18    0
A)             2   0   0   1    0
B             16   0  10  15    0
B)             1   3   0   0    0
C              1   3   0   1    0
C)             0   0   1   0    0
YES            0   0   0   0  337


In [ ]:
task2_final = pd.DataFrame({
    "Model": [
        "Qwen2.5-VL-3B",
        "Qwen2.5-VL-7B"
    ],
    "Samples": [
        len(task1_1000),
        len(qwen7b_1000)
    ],
    "Overall Accuracy (%)": [
        task1_1000["correct"].mean() * 100,
        qwen7b_1000["correct"].mean() * 100
    ],
    "Yes/No Accuracy (%)": [
        task1_1000[
            task1_1000["question_type"].str.lower() == "yes/no"
        ]["correct"].mean() * 100,
        qwen7b_1000[
            qwen7b_1000["question_type"].str.lower() == "yes_no"
        ]["correct"].mean() * 100
    ],
    "Multiple Choice Accuracy (%)": [
        task1_1000[
            task1_1000["question_type"].str.lower() == "multiple choice"
        ]["correct"].mean() * 100,
        qwen7b_1000[
            qwen7b_1000["question_type"].str.lower() == "multiple_choice"
        ]["correct"].mean() * 100
    ]
})

task2_final = task2_final.round(2)

display(task2_final)

,Model,Samples,Overall Accuracy (%),Yes/No Accuracy (%),Multiple Choice Accuracy (%)
0,Qwen2.5-VL-3B,1000,81.8,91.73,71.04
1,Qwen2.5-VL-7B,1000,51.2,35.81,68.21


In [ ]:
task2_final.to_csv(
    "/content/drive/MyDrive/satellite_project/task2_final_comparison.csv",
    index=False
)

print("Saved successfully.")
print(
    "/content/drive/MyDrive/satellite_project/task2_final_comparison.csv"
)

Saved successfully.
/content/drive/MyDrive/satellite_project/task2_final_comparison.csv


In [ ]:
print("=" * 70)
print("TASK 2 FINAL FINDINGS")
print("=" * 70)

print("\nQwen2.5-VL-3B")
print("Overall Accuracy: 81.80%")
print("Yes/No Accuracy: 91.73%")
print("Multiple Choice Accuracy: 71.04%")

print("\nQwen2.5-VL-7B")
print("Overall Accuracy: 51.20%")
print("Yes/No Accuracy: 35.81%")
print("Multiple Choice Accuracy: 68.21%")

print("\nKEY FINDINGS")
print("1. The 3B model achieved higher overall accuracy.")
print("2. Multiple-choice performance was relatively similar.")
print("3. The major performance gap occurred on Yes/No questions.")
print("4. The 7B model showed a strong NO-response bias.")
print("5. Larger model size did not automatically produce better performance in this evaluation.")

TASK 2 FINAL FINDINGS

Qwen2.5-VL-3B
Overall Accuracy: 81.80%
Yes/No Accuracy: 91.73%
Multiple Choice Accuracy: 71.04%

Qwen2.5-VL-7B
Overall Accuracy: 51.20%
Yes/No Accuracy: 35.81%
Multiple Choice Accuracy: 68.21%

KEY FINDINGS
1. The 3B model achieved higher overall accuracy.
2. Multiple-choice performance was relatively similar.
3. The major performance gap occurred on Yes/No questions.
4. The 7B model showed a strong NO-response bias.
5. Larger model size did not automatically produce better performance in this evaluation.


# Gradio

In [ ]:
!pip install -q gradio

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")


PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
!pip install -q gradio qwen-vl-utils transformers accelerate

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

model_3b_name = "Qwen/Qwen2.5-VL-3B-Instruct"

processor_3b = AutoProcessor.from_pretrained(model_3b_name)

model_3b = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_3b_name,
    torch_dtype="auto",
    device_map="auto"
)

print("✅ Qwen2.5-VL-3B loaded successfully!")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

✅ Qwen2.5-VL-3B loaded successfully!


In [ ]:
import time

def run_qwen(model, processor, before_image, after_image, question):

    prompt = f"""
You are analyzing two satellite images of the same location.
The first image is BEFORE.
The second image is AFTER.

Compare the two images and answer the question based only on the changes.

If the question is Yes/No, answer only YES or NO.
If the question is multiple choice, answer only A, B, C, or D.
If the question is not Yes/NO or multiple choice then only explain the answer.

Do not describe the images.

Question:
{question}
"""

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": before_image},
            {"type": "image", "image": after_image},
            {"type": "text", "text": prompt}
        ]
    }]

    image_inputs, video_inputs = process_vision_info(messages)

    text_prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=text_prompt,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    )

    inputs = inputs.to(model.device)

    start_time = time.time()

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=20
        )

    response_time = time.time() - start_time

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(
            inputs.input_ids,
            generated_ids
        )
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True
    )

    answer = output_text[0].strip()

    return answer, response_time

In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()

image_files = list(uploaded.keys())

if len(image_files) != 2:
    print("Please upload exactly 2 images: BEFORE and AFTER")
else:
    before_image = Image.open(image_files[0]).convert("RGB")
    after_image = Image.open(image_files[1]).convert("RGB")

    print("BEFORE:", image_files[0], before_image.size)
    print("AFTER :", image_files[1], after_image.size)

Saving Unknown-2.png to Unknown-2 (2).png
Saving Unknown-3.png to Unknown-3 (1).png
BEFORE: Unknown-2 (2).png (405, 405)
AFTER : Unknown-3 (1).png (405, 405)


In [ ]:
question = "Did any change occur between the BEFORE and AFTER images?"

answer, response_time = run_qwen(
    model_3b,
    processor_3b,
    before_image,
    after_image,
    question
)

print("Question:", question)
print("Answer:", answer)
print("Response time:", round(response_time, 2), "seconds")

Question: Did any change occur between the BEFORE and AFTER images?
Answer: Yes.
Response time: 3.32 seconds


In [ ]:
import gradio as gr

def analyze_image_change(model_choice, before, after, question):

    if before is None or after is None:
        return "Please upload both BEFORE and AFTER images.", ""

    if not question:
        return "Please enter a question.", ""

    if model_choice == "Qwen2.5-VL-3B":
        answer, response_time = run_qwen(
            model_3b,
            processor_3b,
            before,
            after,
            question
        )

    else:
        return f"{model_choice} is not loaded yet.", ""

    return answer, f"{response_time:.2f} seconds"


with gr.Blocks(title="Satellite Change Understanding") as demo:

    gr.Markdown(
        """
        # 🛰️ Satellite Image Change Understanding

        Compare BEFORE and AFTER satellite images using Vision-Language Models.
        """
    )

    with gr.Row():

        before_input = gr.Image(
            type="pil",
            label="BEFORE Image"
        )

        after_input = gr.Image(
            type="pil",
            label="AFTER Image"
        )

    question_input = gr.Textbox(
        label="Question",
        placeholder="Example: Did any change occur between the BEFORE and AFTER images?"
    )

    model_selector = gr.Dropdown(
        choices=[
            "Qwen2.5-VL-3B",
            "Qwen2.5-VL-7B",
            "SmolVLM-500M"
        ],
        value="Qwen2.5-VL-3B",
        label="Select Model"
    )

    analyze_button = gr.Button(
        "🔍 Analyze",
        variant="primary"
    )

    with gr.Row():

        answer_output = gr.Textbox(
            label="Prediction"
        )

        time_output = gr.Textbox(
            label="Response Time"
        )

    analyze_button.click(
        fn=analyze_image_change,
        inputs=[
            model_selector,
            before_input,
            after_input,
            question_input
        ],
        outputs=[
            answer_output,
            time_output
        ]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8d4e297dbca4278725.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
